# 03 - Run Hybrid Models
Train hybrid quantum-classical students (NoKD and KD).

**Fix (2026-03-28):** `quantum_layer.py` now uses a manual batch loop
instead of `TorchLayer`, which avoids the `shape '[32,-1]' is invalid`
RuntimeError in PennyLane 0.38+.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Reload modules in case of cached imports
import importlib
import src.models.quantum_layer
import src.models.student_hybrid
importlib.reload(src.models.quantum_layer)
importlib.reload(src.models.student_hybrid)

from src.utils.seed import set_seed
from src.trainers.train_teacher import train_teacher_cv
from src.trainers.train_student import run_student_cv

set_seed(42)
CSV = os.path.abspath('../data/raw/wdbc.csv')
print('CSV path:', CSV)
print('File exists:', os.path.exists(CSV))

CSV path: /Users/isaac/clawd/research/HybridQMedKD/data/raw/wdbc.csv
File exists: True


## Step 1 - Re-run Teacher (needed for KD logits)

In [2]:
teacher = train_teacher_cv(CSV, pca_dim=4, n_splits=5, seed=42)
print('Teacher training complete.')

[Teacher] Fold 1/5
  AUC=0.9951  F1=0.9231  MCC=0.8759  Time=0.3s
[Teacher] Fold 2/5
  AUC=0.9941  F1=0.9268  MCC=0.8885  Time=0.3s
[Teacher] Fold 3/5
  AUC=0.9845  F1=0.9512  MCC=0.9245  Time=0.3s
[Teacher] Fold 4/5
  AUC=0.9970  F1=0.9438  MCC=0.9119  Time=0.3s
[Teacher] Fold 5/5
  AUC=0.9987  F1=0.9630  MCC=0.9439  Time=0.3s

[Teacher] Mean AUC=0.9939  Mean F1=0.9416
Teacher training complete.


## Step 2 - Student Hybrid NoKD
Quantum layer position: **middle**. No distillation.

In [ ]:
metrics_nokd = run_student_cv(
    CSV,
    teacher_fold_outputs=None,
    model_type='hybrid',
    use_kd=False,
    quantum_position='middle',
    pca_dim=4,
    n_qubits=4,
    n_splits=5,
    seed=42,
    epochs=60,
    lr=1e-3,
    batch_size=16,
    exp_name='student_hybrid_nokd'
)
print('NoKD done.')

[student_hybrid_nokd] Fold 1/5
  AUC=0.9898  F1=0.8941  MCC=0.8313  Train=10673.1s
[student_hybrid_nokd] Fold 2/5


KeyboardInterrupt: 

## Step 3 - Student Hybrid KD
Same architecture, with teacher soft-label distillation (T=2, alpha=0.5).

In [ ]:
metrics_kd = run_student_cv(
    CSV,
    teacher_fold_outputs=teacher,
    model_type='hybrid',
    use_kd=True,
    alpha=0.5,
    T=2.0,
    quantum_position='middle',
    pca_dim=4,
    n_qubits=4,
    n_splits=5,
    seed=42,
    epochs=60,
    lr=1e-3,
    batch_size=16,
    exp_name='student_hybrid_kd'
)
print('KD done.')

[student_hybrid_kd] Fold 1/5


## Results Summary

In [ ]:
import numpy as np

def print_summary(name, metrics):
    for k in ['auc', 'f1', 'mcc', 'acc']:
        vals = [m[k] for m in metrics]
        print(f'[{name}] {k}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}')
    times = [m['train_time'] for m in metrics]
    print(f'[{name}] train_time: {np.mean(times):.1f}s avg per fold')

print_summary('Hybrid-NoKD', metrics_nokd)
print()
print_summary('Hybrid-KD',   metrics_kd)